In [4]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
from IPython.display import display


def find_project_root(start_path: Path) -> Path:
    current_path = start_path.resolve()

    for candidate in [current_path, *current_path.parents]:
        if (candidate / "src" / "config.py").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    GENRES,
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    SAMPLE_RATE,
    DURATION,
)

AUDIO_DIR = Path(RAW_DATA_DIR) / "genres_original"
PROCESSED_DIR = Path(PROCESSED_DATA_DIR)

print("Python:", sys.executable)
print("Project root:", PROJECT_ROOT)
print("Audio directory:", AUDIO_DIR)
print("Dataset exists:", AUDIO_DIR.exists())

if not AUDIO_DIR.exists():
    raise FileNotFoundError(
        f"Dataset not found at:\n{AUDIO_DIR}"
    )

Python: d:\Projects\music-genre-classification-AI\.venv\Scripts\python.exe
Project root: D:\Projects\music-genre-classification-AI
Audio directory: D:\Projects\music-genre-classification-AI\data\raw\genres_original
Dataset exists: True


In [5]:
inventory_records = []

for genre in GENRES:
    genre_folder = AUDIO_DIR / genre

    if not genre_folder.exists():
        print(f"Missing genre folder: {genre}")
        continue

    audio_files = sorted(genre_folder.glob("*.wav"))

    for audio_path in audio_files:
        inventory_records.append(
            {
                "file_path": str(audio_path.relative_to(PROJECT_ROOT)),
                "filename": audio_path.name,
                "genre": genre,
                "size_bytes": audio_path.stat().st_size,
            }
        )

inventory_df = pd.DataFrame(inventory_records)

print("Total WAV files:", len(inventory_df))

display(
    inventory_df["genre"]
    .value_counts()
    .sort_index()
    .rename("file_count")
    .to_frame()
)

Total WAV files: 1000


,file_count
genre,
blues,100
classical,100
country,100
disco,100
hiphop,100
jazz,100
metal,100
pop,100
reggae,100


In [7]:
def validate_audio_file(audio_path: Path) -> dict:
    result = {
        "status": "valid",
        "sample_rate": None,
        "channels": None,
        "frames": None,
        "duration_seconds": None,
        "format": None,
        "subtype": None,
        "peak_amplitude": None,
        "rms": None,
        "warnings": "",
        "error_type": "",
        "error_message": "",
    }

    warnings_list = []

    try:
        audio_info = sf.info(str(audio_path))

        result["sample_rate"] = int(audio_info.samplerate)
        result["channels"] = int(audio_info.channels)
        result["frames"] = int(audio_info.frames)
        result["duration_seconds"] = float(audio_info.duration)
        result["format"] = str(audio_info.format)
        result["subtype"] = str(audio_info.subtype)

        if audio_info.frames <= 0:
            raise ValueError("Audio file has no frames.")

        sum_of_squares = 0.0
        total_samples = 0
        peak_amplitude = 0.0

        with sf.SoundFile(str(audio_path), mode="r") as audio_file:
            while True:
                audio_block = audio_file.read(
                    frames=65536,
                    dtype="float32",
                    always_2d=True,
                )

                if audio_block.size == 0:
                    break

                if not np.isfinite(audio_block).all():
                    raise ValueError(
                        "Audio contains invalid numerical values."
                    )

                peak_amplitude = max(
                    peak_amplitude,
                    float(np.max(np.abs(audio_block))),
                )

                audio_block_64 = audio_block.astype(
                    np.float64,
                    copy=False,
                )

                sum_of_squares += float(
                    np.sum(audio_block_64 ** 2)
                )

                total_samples += audio_block_64.size

        if total_samples == 0:
            raise ValueError("No audio samples were decoded.")

        result["peak_amplitude"] = peak_amplitude
        result["rms"] = float(
            np.sqrt(sum_of_squares / total_samples)
        )

        if result["sample_rate"] != SAMPLE_RATE:
            warnings_list.append(
                f"Sample rate is {result['sample_rate']} Hz"
            )

        if result["channels"] != 1:
            warnings_list.append(
                f"Audio has {result['channels']} channels"
            )

        if abs(result["duration_seconds"] - DURATION) > 1.5:
            warnings_list.append(
                f"Duration is {result['duration_seconds']:.2f} seconds"
            )

        if result["peak_amplitude"] < 0.0001:
            warnings_list.append(
                "Audio may be silent"
            )

        if warnings_list:
            result["status"] = "warning"

    except Exception as error:
        result["status"] = "invalid"
        result["error_type"] = type(error).__name__
        result["error_message"] = str(error)

    result["warnings"] = " | ".join(warnings_list)

    return result

In [8]:
validation_records = []
total_files = len(inventory_df)

for index, row in inventory_df.iterrows():
    relative_path = Path(row["file_path"])
    absolute_path = PROJECT_ROOT / relative_path

    validation_result = validate_audio_file(
        absolute_path
    )

    validation_records.append(
        {
            **row.to_dict(),
            **validation_result,
        }
    )

    completed = index + 1

    if completed % 50 == 0 or completed == total_files:
        print(
            f"Validated {completed}/{total_files} files"
        )

validation_full_df = pd.DataFrame(
    validation_records
)

print("\nValidation completed.")

display(validation_full_df.head())

Validated 50/1000 files
Validated 100/1000 files
Validated 150/1000 files
Validated 200/1000 files
Validated 250/1000 files
Validated 300/1000 files
Validated 350/1000 files
Validated 400/1000 files
Validated 450/1000 files
Validated 500/1000 files
Validated 550/1000 files
Validated 600/1000 files
Validated 650/1000 files
Validated 700/1000 files
Validated 750/1000 files
Validated 800/1000 files
Validated 850/1000 files
Validated 900/1000 files
Validated 950/1000 files
Validated 1000/1000 files

Validation completed.


,file_path,filename,genre,size_bytes,status,sample_rate,channels,frames,duration_seconds,format,subtype,peak_amplitude,rms,warnings,error_type,error_message
0,data\raw\genres_original\blues\blues.00000.wav,blues.00000.wav,blues,1323632,valid,22050.0,1.0,661794.0,30.013333,WAV,PCM_16,0.885376,0.140688,,,
1,data\raw\genres_original\blues\blues.00001.wav,blues.00001.wav,blues,1323632,valid,22050.0,1.0,661794.0,30.013333,WAV,PCM_16,0.683807,0.107619,,,
2,data\raw\genres_original\blues\blues.00002.wav,blues.00002.wav,blues,1323632,valid,22050.0,1.0,661794.0,30.013333,WAV,PCM_16,0.848053,0.183227,,,
3,data\raw\genres_original\blues\blues.00003.wav,blues.00003.wav,blues,1323632,valid,22050.0,1.0,661794.0,30.013333,WAV,PCM_16,0.845886,0.162029,,,
4,data\raw\genres_original\blues\blues.00004.wav,blues.00004.wav,blues,1323632,valid,22050.0,1.0,661794.0,30.013333,WAV,PCM_16,0.859589,0.103356,,,


In [9]:
status_summary = (
    validation_full_df["status"]
    .value_counts()
    .reindex(
        ["valid", "warning", "invalid"],
        fill_value=0,
    )
)

display(
    status_summary
    .rename("file_count")
    .to_frame()
)

genre_summary = pd.crosstab(
    validation_full_df["genre"],
    validation_full_df["status"],
)

display(genre_summary)

,file_count
status,
valid,999
warning,0
invalid,1


status,invalid,valid
genre,,
blues,0,100
classical,0,100
country,0,100
disco,0,100
hiphop,0,100
jazz,1,99
metal,0,100
pop,0,100
reggae,0,100


In [10]:
usable_df = validation_full_df[
    validation_full_df["status"] != "invalid"
].copy()

invalid_df = validation_full_df[
    validation_full_df["status"] == "invalid"
].copy()

print("Usable files:", len(usable_df))
print("Invalid files:", len(invalid_df))

if not invalid_df.empty:
    display(
        invalid_df[
            [
                "filename",
                "genre",
                "error_type",
                "error_message",
            ]
        ]
    )

Usable files: 999
Invalid files: 1


,filename,genre,error_type,error_message
554,jazz.00054.wav,jazz,LibsndfileError,Error opening 'D:\\Projects\\music-genre-class...


In [11]:
PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

validation_output = (
    PROCESSED_DIR / "dataset_validation.csv"
)

full_output = (
    PROCESSED_DIR / "dataset_validation_full.csv"
)

invalid_output = (
    PROCESSED_DIR / "invalid_audio_files.csv"
)

usable_df.to_csv(
    validation_output,
    index=False,
)

validation_full_df.to_csv(
    full_output,
    index=False,
)

invalid_df.to_csv(
    invalid_output,
    index=False,
)

print("Saved:")
print(validation_output)
print(full_output)
print(invalid_output)

Saved:
D:\Projects\music-genre-classification-AI\data\processed\dataset_validation.csv
D:\Projects\music-genre-classification-AI\data\processed\dataset_validation_full.csv
D:\Projects\music-genre-classification-AI\data\processed\invalid_audio_files.csv


In [12]:
saved_df = pd.read_csv(
    validation_output
)

print("File exists:", validation_output.exists())
print("Saved rows:", len(saved_df))
print("Columns:", saved_df.columns.tolist())

display(saved_df.head())

File exists: True
Saved rows: 999
Columns: ['file_path', 'filename', 'genre', 'size_bytes', 'status', 'sample_rate', 'channels', 'frames', 'duration_seconds', 'format', 'subtype', 'peak_amplitude', 'rms', 'warnings', 'error_type', 'error_message']


,file_path,filename,genre,size_bytes,status,sample_rate,channels,frames,duration_seconds,format,subtype,peak_amplitude,rms,warnings,error_type,error_message
0,data\raw\genres_original\blues\blues.00000.wav,blues.00000.wav,blues,1323632,valid,22050.0,1.0,661794.0,30.013333,WAV,PCM_16,0.885376,0.140688,NaN,NaN,NaN
1,data\raw\genres_original\blues\blues.00001.wav,blues.00001.wav,blues,1323632,valid,22050.0,1.0,661794.0,30.013333,WAV,PCM_16,0.683807,0.107619,NaN,NaN,NaN
2,data\raw\genres_original\blues\blues.00002.wav,blues.00002.wav,blues,1323632,valid,22050.0,1.0,661794.0,30.013333,WAV,PCM_16,0.848053,0.183227,NaN,NaN,NaN
3,data\raw\genres_original\blues\blues.00003.wav,blues.00003.wav,blues,1323632,valid,22050.0,1.0,661794.0,30.013333,WAV,PCM_16,0.845886,0.162029,NaN,NaN,NaN
4,data\raw\genres_original\blues\blues.00004.wav,blues.00004.wav,blues,1323632,valid,22050.0,1.0,661794.0,30.013333,WAV,PCM_16,0.859589,0.103356,NaN,NaN,NaN
